In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F 
import random
import numpy as np
from torch_geometric.transforms import NormalizeFeatures

seed = 42

torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)

torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
from torch_geometric.datasets import Planetoid

In [4]:

dataset = Planetoid(root="data", name="Cora", transform=NormalizeFeatures())
data = dataset[0]

In [5]:
class GraphSAGESampler:
  def __init__(self, adj):
    self.adj = adj
 
  def sample_neighbors(self, nodes, num_samples):

    neigh = self.adj[nodes]  # [B, max_degree]

    if neigh.shape[1] >= num_samples:
        perm = torch.rand(neigh.shape).argsort(dim=1)
        neigh = torch.gather(neigh, 1, perm[:, :num_samples])
    else:
        idx = torch.randint(0, neigh.shape[1], (nodes.shape[0], num_samples))
        neigh = torch.gather(neigh, 1, idx)

    return neigh
  
  def sample(self, batch_nodes, num_samples_list):
    layers = [batch_nodes]
    current = batch_nodes


    for num_samples in num_samples_list:
      neigh = self.sample_neighbors(current, num_samples)

      current = neigh.reshape(-1) # flattening

      layers.append(current)

    return layers

In [25]:
class MeanAggregator(nn.Module):

  def __init__(self, input_dim, output_dim, dropout=0.5, bias=False, concat=False):
    super().__init__()

    self.dropout = dropout
    self.concat = concat

    self.self_weights = nn.Linear(input_dim, output_dim, bias=bias)
    self.neigh_weights = nn.Linear(input_dim, output_dim, bias=bias)

    self.act = nn.ReLU()

  def forward(self, self_vecs, neigh_vecs):
    """
    self_vecs:  [B, D]
    neigh_vecs: [B, S, D]
    """

    self_vecs = F.dropout(self_vecs, self.dropout, self.training)
    neigh_vecs = F.dropout(neigh_vecs, self.dropout, self.training)

    neigh_mean = neigh_vecs.mean(dim=1)   # [B, D]

    self_part = self.self_weights(self_vecs)
    neigh_part = self.neigh_weights(neigh_mean)

    if self.concat:
        out = torch.cat([self_part, neigh_part], dim=1)
    else:
        out = self_part + neigh_part

    return self.act(out)

# ------------------- MeanAggregator with He(Kaiming initialization) initialization
# class MeanAggregator(nn.Module):

#     def __init__(self, input_dim, output_dim, dropout=0.5, bias=False, concat=False):
#         super().__init__()

#         self.dropout = dropout
#         self.concat = concat

#         self.self_weights = nn.Linear(input_dim, output_dim, bias=bias)
#         self.neigh_weights = nn.Linear(input_dim, output_dim, bias=bias)

#         # He Initialization
#         nn.init.kaiming_uniform_(self.self_weights.weight, nonlinearity='relu')
#         nn.init.kaiming_uniform_(self.neigh_weights.weight, nonlinearity='relu')

#         if bias:
#             nn.init.zeros_(self.self_weights.bias)
#             nn.init.zeros_(self.neigh_weights.bias)

#         self.act = nn.ReLU()

#     def forward(self, self_vecs, neigh_vecs):
#         """
#         self_vecs:  [B, D]
#         neigh_vecs: [B, S, D]
#         """

#         self_vecs = F.dropout(self_vecs, self.dropout, self.training)
#         neigh_vecs = F.dropout(neigh_vecs, self.dropout, self.training)

#         neigh_mean = neigh_vecs.mean(dim=1)   # [B, D]

#         self_part = self.self_weights(self_vecs)
#         neigh_part = self.neigh_weights(neigh_mean)

#         if self.concat:
#             out = torch.cat([self_part, neigh_part], dim=1)
#         else:
#             out = self_part + neigh_part

#         return self.act(out)
    
class MaxPoolAggregator(nn.Module):
  def __init__(self, input_dim, output_dim, hidden_dim = 256):
    super().__init__()
    self.project_to_hidden = nn.Linear(input_dim, hidden_dim)
    self.project_back = nn.Linear(hidden_dim, output_dim)
    self.project_self_vecs = nn.Linear(input_dim, output_dim)
    self.act = nn.ReLU()

  def forward(self, self_vecs, neigh_vecs):
    '''
    project -> pool -> project_back -> concat
    '''
    neigh_projected = self.act(self.project_to_hidden(neigh_vecs))
    self_projected = self.project_self_vecs(self_vecs)
    neighs_pooled = torch.max(neigh_projected, dim = 1).values
    neighs_pooled_projected_back = self.project_back(neighs_pooled)

    concatinated = self_projected + neighs_pooled_projected_back

    return self.act(concatinated)
    

In [7]:
class GraphSAGE(nn.Module):

    def __init__(self, features, sampler, aggregators, num_samples):

        super().__init__()

        self.features = features
        self.sampler = sampler
        self.aggregators = nn.ModuleList(aggregators)
        self.num_samples = num_samples

    def forward(self, nodes):

        layers = self.sampler.sample(nodes, self.num_samples)

        hidden = [self.features[layer] for layer in layers]

        num_layers = len(self.aggregators)

        for layer in range(num_layers):

            aggregator = self.aggregators[layer]
            next_hidden = []

            for hop in range(num_layers - layer):

                parent = hidden[hop]
                neigh = hidden[hop + 1]

                num_neigh = self.num_samples[hop]  

                feat_dim = neigh.shape[1]

                neigh = neigh.view(parent.shape[0], num_neigh, feat_dim)

                out = aggregator(parent, neigh)

                next_hidden.append(out)

            hidden = next_hidden

        return hidden[0]

In [9]:
class SAGEClassifier(nn.Module):

    def __init__(self, graph_sage, hidden_dim, num_classes,dropout = 0.5):

        super().__init__()

        self.graph_sage = graph_sage
        self.classifier = nn.Linear(hidden_dim, num_classes)
        self.dropout = nn.Dropout(dropout)


    def forward(self, nodes):
        h = self.graph_sage(nodes)
        h = F.normalize(h, p=2, dim = 1)
        h = self.dropout(h)
        return self.classifier(h)

In [10]:
import random

def build_adj_matrix(edge_index, num_nodes, max_degree):

    adj = torch.zeros((num_nodes, max_degree), dtype=torch.long)

    neighbors = [[] for _ in range(num_nodes)]

    src, dst = edge_index

    for u, v in zip(src.tolist(), dst.tolist()):
        neighbors[u].append(v)
        neighbors[v].append(u)

    for i in range(num_nodes):

        neigh = neighbors[i]

        if len(neigh) == 0:
            continue

        if len(neigh) >= max_degree:
            neigh = random.sample(neigh, max_degree)
        else:
            while len(neigh) < max_degree:
                neigh.append(random.choice(neigh))

        adj[i] = torch.tensor(neigh)

    return adj

adj = build_adj_matrix(data.edge_index, data.num_nodes, max_degree=336)
adj.shape

torch.Size([2708, 336])

In [ ]:
adj_list = [[] for _ in range(data.num_nodes)]

src, dst = data.edge_index
for u,v in zip(src.tolist(), dst.tolist()):
  adj_list[u].append(v)
  adj_list[v].append(u)
adj_list

In [26]:
sampler = GraphSAGESampler(adj)

agg1 = MeanAggregator(1433, 256)
agg2 = MeanAggregator(256, 128)

sage = GraphSAGE(
    features=data.x,
    sampler=sampler,
    aggregators=[agg1, agg2],
    num_samples=[25, 10]
)

classifier_cls = SAGEClassifier(
    sage,
    hidden_dim=128,
    num_classes=dataset.num_classes
)
nn.utils.clip_grad_norm_(classifier_cls.parameters(), 5)

tensor(0.)

In [27]:
optimizer = torch.optim.Adam(classifier_cls.parameters(), lr=0.01, weight_decay= 5e-4)
criterion = nn.CrossEntropyLoss()

def train():

    classifier_cls.train()

    optimizer.zero_grad()

    train_nodes = torch.where(data.train_mask)[0]

    logits = classifier_cls(train_nodes)

    loss = criterion(logits, data.y[train_nodes])

    loss.backward()
    
    optimizer.step()

    return loss.item()

In [28]:
for epoch in range(200):
  loss = train()
  if epoch % 10 == 0:
    print(f"Epoch {epoch}, loss: {loss: }")

Epoch 0, loss:  1.951851725578308
Epoch 10, loss:  1.2140867710113525
Epoch 20, loss:  0.7240487933158875
Epoch 30, loss:  0.43519988656044006
Epoch 40, loss:  0.2439812570810318
Epoch 50, loss:  0.17414388060569763
Epoch 60, loss:  0.11555959284305573
Epoch 70, loss:  0.0972287654876709
Epoch 80, loss:  0.08622274547815323
Epoch 90, loss:  0.06554944813251495
Epoch 100, loss:  0.055170513689517975
Epoch 110, loss:  0.06484472006559372
Epoch 120, loss:  0.04977719858288765
Epoch 130, loss:  0.04257698729634285
Epoch 140, loss:  0.044978268444538116
Epoch 150, loss:  0.042834024876356125
Epoch 160, loss:  0.04325828701257706
Epoch 170, loss:  0.03328629583120346
Epoch 180, loss:  0.04212174564599991
Epoch 190, loss:  0.035882554948329926


In [ ]:
from sklearn.metrics import classification_report

def test(mask):

    classifier_cls.eval()

    with torch.no_grad():

        nodes = torch.where(mask)[0]

        logits = classifier_cls(nodes)

        pred = logits.argmax(dim=1)

        correct = (pred == data.y[nodes]).sum().item()

        acc = correct / nodes.shape[0]

    return acc

def classification_metrics(mask):

    classifier_cls.eval()

    with torch.no_grad():

        nodes = torch.where(mask)[0]

        logits = classifier_cls(nodes)

        preds = logits.argmax(dim=1).cpu()
        labels = data.y[nodes].cpu()

    print(classification_report(labels, preds))

train_acc = test(data.train_mask)
val_acc = test(data.val_mask)
test_acc = test(data.test_mask)

print("Train Accuracy:", train_acc)
print("Validation Accuracy:", val_acc)
print("Test Accuracy:", test_acc) 
classification_metrics(data.test_mask)

Train Accuracy: 1.0
Validation Accuracy: 0.746
Test Accuracy: 0.764
              precision    recall  f1-score   support

           0       0.62      0.62      0.62       130
           1       0.69      0.84      0.76        91
           2       0.84      0.90      0.87       144
           3       0.87      0.72      0.79       319
           4       0.81      0.79      0.80       149
           5       0.78      0.76      0.77       103
           6       0.58      0.88      0.70        64

    accuracy                           0.77      1000
   macro avg       0.74      0.79      0.76      1000
weighted avg       0.78      0.77      0.77      1000

